EXERCICE 2 : Early stopping et régularisation sur la sévérité sinistre
=======================================================================
Objectif : Appliquer les stratégies anti-overfitting (early stopping, 
régularisation, tuning) sur un problème de coût moyen des sinistres.

Ce que vous allez apprendre :
- Configurer l'early stopping avec XGBoost et LightGBM
- Observer l'impact des hyperparamètres de régularisation
- Diagnostiquer l'overfitting avec les courbes train/val
- Trouver la configuration optimale par tuning

Pré-requis : M03_F05 (matin), Bloc 1-2 de ce module
Temps estimé : 15 minutes

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb

## 1. CHARGEMENT DES DONNÉES — SÉVÉRITÉ SINISTRE

In [ ]:
# =============================================================
# 1. DONNÉES
# =============================================================
print("=" * 60)
print("EXERCICE 2 : Early stopping et régularisation — Sévérité")
print("=" * 60)

# Créer un dataset de sévérité à partir de freMTPL2freq
# (on simule la sévérité car freMTPL2sev n'est pas toujours disponible localement)
df = pd.read_csv("../M03_F01_Machine_Learning_Fondamentaux/freMTPL2freq.csv")

# Ne garder que les contrats avec sinistre (sévérité conditionnelle à avoir un sinistre)
df_sinistres = df[df['ClaimNb'] > 0].copy()

# Simuler un coût moyen de sinistre réaliste (log-normal)
np.random.seed(42)
n_sin = len(df_sinistres)
log_cost = (
    5.5  # base (≈ exp(5.5) = 245€)
    + 0.01 * df_sinistres['DrivAge'].values
    + 0.005 * df_sinistres['BonusMalus'].values
    + 0.02 * df_sinistres['VehAge'].values
    + np.random.normal(0, 0.8, n_sin)
)
df_sinistres['ClaimCost'] = np.exp(log_cost)

features = ['DrivAge', 'BonusMalus', 'VehAge', 'Density']
X = df_sinistres[features].astype(float)
y = df_sinistres['ClaimCost'].astype(float)

print(f"Sinistres avec coût : {len(df_sinistres)} observations")
print(f"Coût moyen : {y.mean():.0f}€ | Médiane : {y.median():.0f}€ | Max : {y.max():.0f}€")

# Split train / validation / test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)
print(f"Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")

In [ ]:
# =============================================================
# 2. XGBOOST SANS EARLY STOPPING (OVERFITTING VOLONTAIRE)
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 1 : XGBoost SANS early stopping (observer l'overfitting)")
print("=" * 60)

# TODO : Entraîner un XGBoost avec beaucoup d'arbres SANS early stopping
# Configuration volontairement "overfit" :
#   n_estimators=2000, learning_rate=0.1, max_depth=8
# Enregistrer les scores train et val à chaque itération avec eval_set



# TODO : Tracer les courbes train/val pour visualiser l'overfitting
# À quel nombre d'arbres commence l'overfitting ?

In [ ]:
# =============================================================
# 3. XGBOOST AVEC EARLY STOPPING
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 2 : XGBoost AVEC early stopping")
print("=" * 60)

# TODO : Même configuration mais avec early_stopping_rounds=50
# Combien d'arbres sont retenus ?



# TODO : Comparer les scores test avec et sans early stopping

In [ ]:
# =============================================================
# 4. IMPACT DES HYPERPARAMÈTRES DE RÉGULARISATION
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 3 : Impact des hyperparamètres de régularisation")
print("=" * 60)

# TODO : Tester différentes valeurs de max_depth et observer l'impact
# max_depths = [2, 3, 4, 5, 6, 8, 10]
# Pour chaque valeur, entraîner avec early stopping et noter le score test



# TODO : Tester différentes valeurs de reg_lambda
# reg_lambdas = [0, 0.1, 1.0, 5.0, 10.0, 50.0]



# TODO : Tester différentes valeurs de subsample
# subsamples = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

In [ ]:
# =============================================================
# 4bis. GRIDSEARCHCV (sans early stopping)
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 4bis : GridSearchCV classique (sans early stopping)")
print("=" * 60)

from sklearn.model_selection import GridSearchCV

# TODO: 
# Précédemment, nous avons testé des variations d'UN hyperparamètres en figeant tous les autres.
# Maintenant, testez toutes les combinaisons d'hyperparamètres possibles.
# Pour cela, utilisez GridSearchCV.

# Attention : GridSearchCV ne prend pas en compte l'early stopping : il faut donc tester plusieurs valeurs de n_estimators.
# (pas de mécanisme pour arrêter automatiquement l'entraînement)
# Valeurs identiques à celles testées Partie 4 (max_depth, reg_lambda, subsample)

param_grid_cv = {
    'n_estimators': [2000],
    'max_depth': [2, 3, 4, 5, 6,],
    'reg_lambda': [0.1, 1.0, 10.0, 50.0],
    'subsample': [0.6, 0.7, 0.8, 0.9],
}

# ...
# ... COMPLETER ...
# ...

# GridSearchCV refait sa propre cross-validation.
# Il faut donc combiner X_train et X_val avant de fit le grid_search

# ...
# ... COMPLETER ...
# ...

# TODO : afficher les meilleurs paramètres du grid_search, et les scores associés.
# Afficher également les RMSE

print(f"\nMeilleurs paramètres (GridSearchCV) : {grid_search.best_params_}")
print(f"Meilleur score CV (RMSE) : {-grid_search.best_score_:.2f}")

y_pred_cv = grid_search.best_estimator_.predict(X_test)
rmse_cv = np.sqrt(mean_squared_error(y_test, y_pred_cv))
print(f"\nRMSE test (GridSearchCV, n_estimators={grid_search.best_params_['n_estimators']}) : {rmse_cv:.2f}")
print(f"→ Comparer avec le grid search manuel + early stopping (Partie 3bis)")

In [ ]:
# =============================================================
# 4ter. GRID SEARCH CONJOINT (tous les hyperparamètres, avec early stopping)
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 4ter : Grid search conjoint avec early stopping")
print("=" * 60)

import itertools

# TODO: 
# Précédemment, nous avons testé des variations d'UN hyperparamètres en figeant tous les autres.
# Maintenant, testez toutes les combinaisons d'hyperparamètres possibles.
# On pourrait utiliser GridSearchCV, mais il ne gère pas nativement early_stopping_rounds.
#  Il faut donc boucler manuellement sur toutes les combinaisons
param_grid = {
    'max_depth': [2, 3, 4, 5, 6, 8, 10],
    'reg_lambda': [0, 0.1, 1.0, 5.0, 10.0, 50.0],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
}


keys = list(param_grid.keys())
combinations = list(itertools.product(*param_grid.values()))
print(f"Nombre total de combinaisons à évaluer : {len(combinations)}")

grid_results = []

# ...
# ... COMPLETER ...
# ...

print("\nTop 10 configurations (par RMSE validation) :")

# Meilleure config trouvée par le grid search conjoint
# ...
# ... COMPLETER ...
# ...

# Réentraîner et évaluer sur le test set
# print(f"\nMeilleure configuration : {best_params_grid}")

print(f"→ Comparer avec le tuning séquentiel (one-at-a-time) de la Partie 3")

In [ ]:
# =============================================================
# 5. CONFIGURATION OPTIMALE
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 4 : Configuration optimale")
print("=" * 60)

# TODO : Combiner les meilleurs hyperparamètres trouvés
# et entraîner un modèle final avec early stopping



# TODO : Comparer avec le modèle initial (overfit) sur le test set

## QUESTIONS DE RÉFLEXION

In [ ]:
# =============================================================
# QUESTIONS DE RÉFLEXION
# =============================================================
"""
1. Sans early stopping, à partir de combien d'arbres l'overfitting commence-t-il ?
   Que se passe-t-il si on continue encore 1000 arbres de plus ?

2. L'early stopping a retenu combien d'arbres ? Que dit ce nombre sur la 
   complexité nécessaire du modèle ?

3. Quel hyperparamètre a le plus d'impact sur l'overfitting : 
   max_depth, reg_lambda, ou subsample ?

4. Le modèle "régularisé" est-il significativement meilleur sur le test que 
   le modèle "overfit" ? Quantifiez la différence en termes de RMSE.

5. En assurance, pourquoi est-il particulièrement important de contrôler 
   l'overfitting sur la sévérité ? (Indice : impact sur le provisionnement)
"""